# Roofline Analysis of Weight Syncing Latency in Trellis with TPU Raiden

**Author**: `raiden-weight_sync_analyst`  
**Target Model**: Qwen 3.5 397B (60 Layers, 1,008 Tensors, 792.69 GB Payload)  
**Hardware & System Context**:
- **Trainer**: TPU v7x mesh (`gf=4x4x8`, 32 Hosts, FSDP=64, TP=2, EP=1) with **PCIe Gen5 x16** ($252\text{ GB/s/host}$ physical limit calculated at **60% effective efficiency** $= 151.2\text{ GB/s}$ accounting for tensor size distribution + DMA launch overheads)
- **Samplers**: TPU v7x instances (`gf=2x2x2`, 2 Hosts per instance, Attn_DP=8, TP=2, EP=1) with **PCIe Gen5 x16**
- **Network**: Trainer and Sampler VM gVNICs (default 200 Gbps) calculated at **85% of line rate** ($170\text{ Gbps} = 21.25\text{ GB/s}$)

---

## Weight Sync Pipeline & Bandwidth-Matched Roofline Equations

$$\begin{aligned}
N_{\text{seed}} &= \max\left(1,\; \left\lfloor \frac{H_{\text{train}} \cdot B_{\text{train, TX}}}{H_{\text{sample}} \cdot B_{\text{sample, RX}}} \right\rfloor\right) \\
T_{\text{D2H}} &= \frac{V_{\text{D2H, host}}}{\eta_{\text{D2H}} \cdot BW_{\text{PCIe, D2H}}} + T_{\text{batch\_barrier}} + (N_{\text{variables}} \times \tau_{\text{dma\_launch}}) \quad (\text{PCIe Gen5 x16 @ 60\%} = 151.2\text{ GB/s}) \\
T_{\text{control}} &= T_{\text{RPC, base}} + \alpha \cdot (H_{\text{train}} + N_{\text{samplers}} \cdot H_{\text{sample}}) + T_{\text{C++ unroll}} \\
T_{\text{H2H, direct}} &= \max\left(1,\; \frac{N_{\text{samplers}}}{N_{\text{seed}}}\right) \cdot \frac{V_{\text{H2D, host}}}{B_{\text{sample, RX}}} \\
T_{\text{H2H, matched}} &= \frac{V_{\text{H2D, host}}}{B_{\text{sample}}} + \left(\left\lceil \frac{N_{\text{samplers}}}{N_{\text{seed}}} \right\rceil - 1\right) \cdot \Delta t_{\text{chunk}} \\
T_{\text{tail, H2D}} &= \max\left(0, \frac{V_{\text{H2D, host}}}{BW_{\text{tile}}} - T_{\text{H2H}}\right) + \frac{V_{\text{H2D, host}} / M}{\eta_{\text{H2D}} \cdot BW_{\text{PCIe, H2D}}} + T_{\text{chunk\_launch}} \\
T_{\text{total}} &= T_{\text{D2H}} + T_{\text{control}} + T_{\text{H2H}} + T_{\text{tail, H2D}} + T_{\text{KV\_reset}}
\end{aligned}$$

In [ ]:
import dataclasses
import math

import matplotlib.pyplot as plt
import numpy as np

# Core Roofline Model Definitions
from google3.third_party.tpu_raiden.tpu_sync.weight_sync.analysis.roofline_model import (
    ModelSpec,
    QWEN_3_5_397B_SPEC,
    RooflineModel,
    SamplerTopology,
    TrainerTopology,
    TransferStrategy,
)

print(f"Loaded Roofline Model for: {QWEN_3_5_397B_SPEC.name}")
print(
    f"Total Payload: {QWEN_3_5_397B_SPEC.total_bytes / 1e9:.2f} GB"
    f" ({QWEN_3_5_397B_SPEC.num_variables} variables across"
    f" {QWEN_3_5_397B_SPEC.num_layers} layers)"
)

## Section 1: Stage-by-Stage Breakdown & Comparison against Empirical Ground Truth (b/558046180)

We evaluate the steady-state performance on $N_{\text{samplers}}=1$ with 200 Gbps NICs and 32 Trainer hosts.


In [ ]:
trainer = TrainerTopology(num_hosts=32, nic_egress_bw_gbps=200.0)
sampler = SamplerTopology(
    num_samplers=1, hosts_per_sampler=2, nic_ingress_bw_gbps=200.0
)
strategy = TransferStrategy(
    enable_relay_broadcast=True, verify_checksums=False, pipelined=True
)

res = RooflineModel.evaluate(
    model=QWEN_3_5_397B_SPEC,
    trainer=trainer,
    sampler=sampler,
    strategy=strategy,
)

stages = [
    "1. D2H DMA",
    "2. Control Plane",
    "3. Physical H2H",
    "4. Exposed H2D Tail",
    "5. KV Reset",
]
times = [
    res.d2h_s,
    res.control_plane_s,
    res.h2h_wire_s,
    res.h2d_exposed_tail_s,
    res.kv_reset_s,
]

plt.figure(figsize=(10, 5))
bars = plt.bar(
    stages, times, color=["#4285F4", "#EA4335", "#FBBC04", "#34A853", "#9AA0A6"]
)
plt.ylabel("Time (seconds)")
plt.title(
    f"Weight Sync Latency Breakdown: {QWEN_3_5_397B_SPEC.name} (Total:"
    f" {res.total_s:.2f}s, Roofline: {res.hardware_roofline_floor_s:.2f}s)"
)
plt.grid(axis="y", linestyle="--", alpha=0.7)

for bar in bars:
  yval = bar.get_height()
  plt.text(
      bar.get_x() + bar.get_width() / 2.0,
      yval + 0.1,
      f"{yval:.2f}s",
      ha="center",
      va="bottom",
      fontweight="bold",
  )

plt.tight_layout()
plt.show()

print(f"Total Sync Time:          {res.total_s:.2f} s")
print(
    f"Hardware Roofline Floor:  {res.hardware_roofline_floor_s:.2f} s"
    f" ({res.efficiency_vs_roofline:.1f}% efficiency)"
)
print(
    f"Primary Bottleneck:       {res.bottleneck_stage}"
    f" ({res.bottleneck_description})"
)

## Section 2: Scalability Sweeps — Direct P2P vs. Bandwidth-Matched Relay

We analyze the weight synchronization scaling characteristics as the number of inference samplers scales from $N=1$ to $N=128$ under Trainer and Sampler host NIC bandwidth constraints, comparing Direct P2P (Unrelayed), Bandwidth-Matched Store-and-Forward Doubling, and Bandwidth-Matched Pipelined Relay.

In [ ]:
sampler_counts = [1, 2, 4, 8, 16, 32, 64, 128]
sweeps = RooflineModel.sweep_samplers(
    model=QWEN_3_5_397B_SPEC,
    trainer=trainer,
    sampler_base=sampler,
    sampler_counts=sampler_counts,
)

direct_times = [sweeps['direct_p2p'][n].total_s for n in sampler_counts]
sf_times = [
    sweeps['matched_store_and_forward'][n].total_s for n in sampler_counts
]
pipelined_times = [
    sweeps['matched_pipelined'][n].total_s for n in sampler_counts
]

plt.figure(figsize=(11, 6))
plt.plot(
    sampler_counts,
    direct_times,
    'o-',
    linewidth=2.5,
    color='#EA4335',
    label='Direct P2P (Unrelayed)',
)
plt.plot(
    sampler_counts,
    sf_times,
    's-',
    linewidth=2.5,
    color='#FBBC04',
    label='Bandwidth-Matched Store-and-Forward',
)
plt.plot(
    sampler_counts,
    pipelined_times,
    '^-',
    linewidth=2.5,
    color='#34A853',
    label='Bandwidth-Matched Pipelined Relay',
)

plt.xscale('log', base=2)
plt.xticks(sampler_counts, [str(n) for n in sampler_counts])
plt.xlabel('Number of Sampler Instances ($N_{samplers}$)', fontsize=12)
plt.ylabel('Total Weight Sync Latency (seconds)', fontsize=12)
plt.title(
    'Scalability Comparison: Direct P2P vs. Bandwidth-Matched Relay'
    f' ({QWEN_3_5_397B_SPEC.name}, 200G Trainer & Sampler NICs)',
    fontsize=13,
)
plt.grid(True, which='both', ls='--', alpha=0.6)
plt.legend(fontsize=11)

speedup = direct_times[-1] / pipelined_times[-1]
plt.annotate(
    f'{speedup:.2f}× Speedup\n(Direct: {direct_times[-1]:.1f}s vs Matched'
    f' Pipelined: {pipelined_times[-1]:.1f}s)',
    xy=(128, pipelined_times[-1]),
    xytext=(32, 60),
    arrowprops=dict(facecolor='black', shrink=0.05, width=1.5, headwidth=8),
    fontweight='bold',
    fontsize=11,
)

plt.tight_layout()
plt.show()